In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


After mounting your Google Drive, you can navigate to the folder where your TXT files are located. You can list the contents of a directory using `!ls` to find the path to your TXT files. For a folder shared directly with your account, it often appears under `MyDrive`. If it's a "Shared Drive" (formerly Team Drive), it would be under `Shareddrives`. Let's start by listing the contents of your `MyDrive` folder.

# Replace 'my_data' with the actual path to your folder in Google Drive
# First, check your MyDrive for shared folders:
!ls /content/drive/MyDrive/'Machine Learning'/ProyectoML

# If it's a 'Shared Drive' (formerly Team Drive), you would look here:
#!ls /content/drive/Shareddrives/'Proyecto ML'

Once you have the path to your TXT file, you can read it into a pandas DataFrame. The `pd.read_csv()` function is versatile enough to handle TXT files, where you can specify the delimiter. For example, if your file is comma-separated, tab-separated, or space-separated:

import pandas as pd

file_path = '/content/drive/MyDrive/Machine Learning/ProyectoML/Copia de SECRETO_oficio_dando_cuenta_toma_de_declaración.txt'

# First, let's try to read the file line by line to inspect its content
# This helps us identify the actual delimiter or if it's unstructured text.
print(f"Reading the first 10 lines of: {file_path}\n")
lines = []
with open(file_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        lines.append(line.strip())
        if i >= 9: # Print first 10 lines
            break

#for i, line in enumerate(lines):
#    print(f"Line {i+1}: {line}")

# Based on the inspection, you can then try different delimiters for pd.read_csv.
# Common delimiters are ',' for CSV, '\t' for tab-separated, or ' ' for space-separated.
# If it's unstructured text, you might need a different approach,
# e.g., reading it as a single string or line by line and then processing it.

# Since this is plain text without clear delimiters, we'll read each line as an entry in a single column:
df_plain_text = pd.DataFrame(lines, columns=['text_content'])
display(df_plain_text.head())

In [3]:
import pandas as pd
import os

First, you need to define the path to the directory containing all your TXT files. Make sure this path is correct for your Google Drive setup.

In [4]:
# Replace this with the actual path to your folder containing TXT files
directory_path = '/content/drive/MyDrive/Machine Learning/ProyectoML'

# Get a list of all .txt files in the directory
txt_files = [f for f in os.listdir(directory_path) if f.endswith('.txt')]

Now, we will loop through each identified TXT file, read its content line by line (as we determined is suitable for unstructured text), and combine all the data into a single DataFrame.

In [6]:
all_data = []

for file_name in txt_files:
    file_path = os.path.join(directory_path, file_name)
    #print(f"Processing file: {file_name}")
    lines = []
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                lines.append(line.strip())
        # Create a temporary DataFrame for the current file
        df_temp = pd.DataFrame(lines, columns=['text_content'])
        # Add a column to identify the source file
        df_temp['source_file'] = file_name
        all_data.append(df_temp)
    except Exception as e:
        print(f"Error reading {file_name}: {e}")

# Concatenate all individual DataFrames into one large DataFrame
if all_data:
    df_combined = pd.concat(all_data, ignore_index=True)
    print(f"\nSuccessfully combined {len(txt_files)} files into a single DataFrame with {len(df_combined)} rows.")
    display(df_combined.head())
else:
    print("No TXT files found or processed successfully.")


Successfully combined 157 files into a single DataFrame with 32295 rows.


,text_content,source_file
0,Ihns Sr. Don.,Copia de Carta_de_José_Cortina_Prieto_para_Em...
1,Emilio Manglaro,Copia de Carta_de_José_Cortina_Prieto_para_Em...
2,,Copia de Carta_de_José_Cortina_Prieto_para_Em...
3,JOSE CORTINA PRIETO,Copia de Carta_de_José_Cortina_Prieto_para_Em...
4,,Copia de Carta_de_José_Cortina_Prieto_para_Em...


In [7]:
import re

# 1. Calculate 'número de caracteres' for each line and also for the full text of each file
df_combined['char_count_line'] = df_combined['text_content'].str.len()

# 2. Extract 'asunto' (subject)
# Look for lines starting with 'ASUNTO:' case-insensitively
asunto_pattern = re.compile(r'^ASUNTO:\s*(.*)', re.IGNORECASE)

def extract_asunto(text):
    match = asunto_pattern.match(text)
    if match:
        return match.group(1).strip()
    return None

# Apply the function to find all potential 'asunto' lines
df_asuntos = df_combined[df_combined['text_content'].str.contains('ASUNTO:', case=False, na=False)].copy()
df_asuntos['asunto'] = df_asuntos['text_content'].apply(extract_asunto)

# Keep only the first 'asunto' found for each file, if multiple exist
df_asuntos_unique = df_asuntos.dropna(subset=['asunto']).groupby('source_file')['asunto'].first().reset_index()

# 3. Aggregate data per file for 'texto completo' and total 'número de caracteres'
df_summary = df_combined.groupby('source_file').agg(
    texto_completo=('text_content', lambda x: '\n'.join(x)), # Join all lines with a newline
    numero_de_caracteres=('char_count_line', 'sum') # Sum character count for each file
).reset_index()

# 4. Merge 'asunto' information into the summary DataFrame
df_final_table = pd.merge(df_summary, df_asuntos_unique, on='source_file', how='left')

# 5. Rename columns to match the user's request
df_final_table = df_final_table.rename(columns={
    'source_file': 'nombre del archivo',
    'asunto': 'asunto',
    'numero_de_caracteres': 'número de caracteres',
    'texto_completo': 'texto completo'
})

# Display the final table
display(df_final_table.head())

,nombre del archivo,texto completo,número de caracteres,asunto
0,Copia de Actitud_del_CESID_ante_la_situación_...,ACTUACION DEL CESID ANTE LA SITUACION PROVOCAD...,16287,NaN
1,Copia de Ambiente_en_los_cuarteles_(25_de_febr...,"Madrid, 25 de Febrero de 1982\n\nEmilio:\n\nPo...",3410,NaN
2,Copia de Capitán_Sánchez_Valiente_(2_de_juni...,88\n\nN.Rfa.: C/D13/10544/O2-06-81\n\nCalifica...,1511,CAPITAN SANCHEZ VALIENTE.
3,Copia de Carta_de_José_Cortina_Prieto_para_Em...,Ihns Sr. Don.\nEmilio Manglaro\n\nJOSE CORTINA...,1437,NaN
4,Copia de Certificación_solicitada_para_su_uni...,MINISTERIO DE DEFENSA\nCENTRO SUPERIOR\nDE\nIN...,2422,CERTIFICACION SOLICITADA PARA SU UNION A LA CA...


In [8]:
df_final_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157 entries, 0 to 156
Data columns (total 4 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   nombre del archivo    157 non-null    object
 1   texto completo        157 non-null    object
 2   número de caracteres  157 non-null    int64 
 3   asunto                85 non-null     object
dtypes: int64(1), object(3)
memory usage: 5.0+ KB
